# ==========================================================
# 03_clean_reconcile.ipynb - Business Cleaning Rules
# ==========================================================

In [ ]:
import pathlib
import pandas as pd

In [ ]:
PROJECT_ROOT = pathlib.Path.cwd().parent
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
RAW_DIR = PROJECT_ROOT / "data" / "raw"

In [ ]:
# Load cleaned bookings dataset from notebook 2
df_bookings = pd.read_csv(CLEAN_DIR / "bookings_merged.csv.gz", parse_dates=["created_at", "slot_at"])
print(f"Loaded merged bookings: {len(df_bookings):,} rows")

In [ ]:
# ----------------------------------------------------------
# Step 1: Merge Collections Status
# ----------------------------------------------------------
collections_files = sorted((RAW_DIR / "collections").glob("*.csv.gz"))
list_coll = [pd.read_csv(f, dtype=str) for f in collections_files]
df_collections = pd.concat(list_coll, ignore_index=True).drop_duplicates(subset=["booking_id"])

df_bookings = df_bookings.merge(
    df_collections[["booking_id", "status"]].rename(columns={"status": "coll_status"}),
    on="booking_id",
    how="left"
)
df_bookings["coll_status"] = df_bookings["coll_status"].fillna("no_collection_row")

In [ ]:
# ----------------------------------------------------------
# Step 2: Apply Cleaning Rules & Filter Out Rejects
# ----------------------------------------------------------
# Rule 1: Exclude rescheduled original Nova bookings
mask_rescheduled = df_bookings.get("status_at_export", pd.Series(dtype="str")).eq("rescheduled").fillna(False)

# Rule 2: Exclude empty test placeholders
mask_empty_tests = (df_bookings["system"] == "nova") & df_bookings["test_code"].isna() & df_bookings["test_price"].isna()

In [ ]:
reject_mask = mask_rescheduled | mask_empty_tests

df_clean_bookings = df_bookings[~reject_mask].copy()
df_rejects = df_bookings[reject_mask].copy()

print(f"Total Rows In: {len(df_bookings):,}")
print(f"Total Rows Rejected: {len(df_rejects):,}")
print(f"Total Clean Rows Remaining: {len(df_clean_bookings):,}")

In [ ]:
# Save fully processed clean bookings
df_clean_bookings.to_csv(CLEAN_DIR / "bookings_clean.csv.gz", index=False, compression="gzip")
df_rejects.to_csv(CLEAN_DIR / "rejects_step5.csv.gz", index=False, compression="gzip")
print("Saved bookings_clean.csv.gz and rejects_step5.csv.gz")